# K-EmoCon Baseline — Full Pipeline (Kaggle GPU)

Runs the complete cross-person affect prediction pipeline on Kaggle with GPU support for the LSTM model.

**Before running:**
1. Attach the **K-EmoCon data** dataset (contains `e4_data/`, `neurosky_polar_data/`, `emotion_annotations/`, `metadata/`)
2. Attach the **kemocon-baseline** code dataset (your uploaded repo zip)
3. Enable GPU accelerator (T4 recommended)
4. Enable internet access (for pip installs)

**Expected Kaggle input paths:**
- Code:  `/kaggle/input/kemocon-baseline/`
- Data:  `/kaggle/input/kemocon-dataset/`  (adjust `DATA_DATASET_SLUG` below if different)

In [ ]:
# ── 0. Configuration ──────────────────────────────────────────────────────
# Adjust these slugs to match your Kaggle dataset names
CODE_DATASET_SLUG = 'kemocon-baseline'   # slug of your uploaded code dataset
DATA_DATASET_SLUG = 'kemocon-dataset'    # slug of the K-EmoCon data dataset

import os
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

CODE_SRC = KAGGLE_INPUT / CODE_DATASET_SLUG
DATA_SRC = KAGGLE_INPUT / DATA_DATASET_SLUG

# Verify datasets are attached
for p, label in [(CODE_SRC, 'Code dataset'), (DATA_SRC, 'Data dataset')]:
    if p.exists():
        print(f'✓ {label}: {p}')
    else:
        candidates = sorted(KAGGLE_INPUT.iterdir()) if KAGGLE_INPUT.exists() else []
        print(f'✗ {label} not found at {p}')
        print(f'  Available datasets: {[d.name for d in candidates]}')

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────
# PyTorch with CUDA is pre-installed on Kaggle GPU instances
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('catboost')
pip('optuna')
pip('opensmile')   # eGeMAPS audio features
pip('scipy', 'statsmodels')

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── 2. Copy code to /kaggle/working ───────────────────────────────────────
import shutil

CODE_DIR = WORK / 'kemocon_baseline'

if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)

# The code dataset may be a flat directory or contain a zip
zip_candidates = list(CODE_SRC.rglob('*.zip'))
if zip_candidates:
    import zipfile
    with zipfile.ZipFile(zip_candidates[0]) as zf:
        zf.extractall(CODE_DIR)
    # If extracted into a subdirectory, find the actual root
    py_files = list(CODE_DIR.rglob('main.py'))
    if py_files:
        actual_root = py_files[0].parent
        if actual_root != CODE_DIR:
            shutil.move(str(actual_root), str(CODE_DIR.parent / '_tmp_code'))
            shutil.rmtree(CODE_DIR)
            shutil.move(str(CODE_DIR.parent / '_tmp_code'), str(CODE_DIR))
else:
    shutil.copytree(CODE_SRC, CODE_DIR)

print('Code directory contents:')
print([p.name for p in sorted(CODE_DIR.iterdir())])

In [ ]:
# ── 3. Patch config.py to point at Kaggle data paths ─────────────────────
config_path = CODE_DIR / 'config.py'
config_text = config_path.read_text()

# Replace DATA_ROOT to point at the attached Kaggle data dataset
patched = config_text.replace(
    'DATA_ROOT = PROJECT_ROOT / "data"',
    f'DATA_ROOT = Path("{DATA_SRC}")'
)

if patched == config_text:
    print('WARNING: could not patch DATA_ROOT — check config.py manually')
else:
    config_path.write_text(patched)
    print(f'config.py patched: DATA_ROOT → {DATA_SRC}')

# Verify the data directories exist
for subdir in ['e4_data', 'neurosky_polar_data', 'emotion_annotations', 'metadata']:
    p = DATA_SRC / subdir
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {status}  {p}')

In [ ]:
# ── 4. Run the pipeline ───────────────────────────────────────────────────
# Flags:
#   --no-optuna   skip CatBoostOptuna (confirmed negligible gain in prior runs)
#   LSTM is enabled by default now; GPU will be used automatically
#
# Add --no-lstm to skip LSTM if you want a quick run:
#   cmd.append('--no-lstm')
# Add --no-controls to skip negative controls (saves ~40% time):
#   cmd.append('--no-controls')

import sys, subprocess, time

LOG = WORK / 'pipeline.log'

cmd = [
    sys.executable, str(CODE_DIR / 'main.py'),
    '--no-optuna',
]

print('Running:', ' '.join(cmd))
print(f'Log: {LOG}\n')

t0 = time.time()
with open(LOG, 'w') as log_fh:
    proc = subprocess.Popen(
        cmd,
        cwd=str(CODE_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
        log_fh.write(line)

proc.wait()
elapsed = time.time() - t0
print(f'\nExit code: {proc.returncode}  |  Elapsed: {elapsed/60:.1f} min')

In [ ]:
# ── 5. Package results for download ──────────────────────────────────────
import zipfile
from pathlib import Path

results_dir = CODE_DIR / 'results'
output_zip = WORK / 'pipeline_results.zip'

if results_dir.exists():
    run_dirs = sorted(results_dir.iterdir())
    print(f'Result runs found: {[d.name for d in run_dirs]}')

    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for run_dir in run_dirs:
            for f in sorted(run_dir.rglob('*')):
                if f.is_file():
                    zf.write(f, f.relative_to(results_dir))

    size_mb = output_zip.stat().st_size / 1024**2
    print(f'Saved {output_zip}  ({size_mb:.1f} MB)')
    print('Download from: Kaggle → Output tab → pipeline_results.zip')
else:
    print('No results directory found — pipeline may have failed. Check pipeline.log')

In [ ]:
# ── 6. Quick sanity check on results ─────────────────────────────────────
import pandas as pd
from pathlib import Path

results_dir = Path('/kaggle/working/kemocon_baseline/results')
run_dirs = sorted(results_dir.iterdir()) if results_dir.exists() else []

if not run_dirs:
    print('No results yet.')
else:
    latest = run_dirs[-1]
    print(f'Latest run: {latest.name}\n')

    summary_csv = latest / 'stat_summary.csv'
    if summary_csv.exists():
        df = pd.read_csv(summary_csv)
        # Show LSTM and CatBoost at lag_0
        mask = (df['condition'] == 'lag_0') & (df['model'].isin(['LSTM', 'CatBoost', 'RidgeCV', 'MeanBaseline']))
        print(df[mask][['model', 'target', 'ccc_mean', 'ccc_median', 'ccc_std']].to_string(index=False))
    else:
        csvs = list(latest.glob('*.csv'))
        print(f'CSVs in run dir: {[c.name for c in csvs]}')